# AI4I 2020 Predictive Maintenance Pipeline

This notebook covers the ETL, train/test split, preprocessing, feature selection, and model training for the AI4I 2020 Predictive Maintenance dataset. Since we are treating this as a time series (UDI as proxy for time), we will employ temporal splitting to avoid data leakage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, mutual_info_classif

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 1. Data ETL

In [ ]:
# Load data
data_path = 'data/01_raw/pred-maintenance/ai4i2020.csv'
df = pd.read_csv(data_path)
display(df.head())
df.info()

## 2. Train/Test Split & Feature-Target Separation
Since the data is sequential (`UDI` acts as a time indicator), the data must not be shuffled to avoid leakage. Instead, we must split the data temporally.

In [ ]:
target_col = 'Machine failure'

# These columns are specific breakdown types and leak the target label.
leakage_cols = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

# UDI and Product ID are not meaningful features for prediction.
id_cols = ['UDI', 'Product ID']

X = df.drop(columns=[target_col] + leakage_cols + id_cols)
y = df[target_col]

# Temporal split without shuffling (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

## 3. Preprocessing
We process train and test sets separately. 'Type' is an ordinal categorical feature (L = Low, M = Medium, H = High).

In [ ]:
numerical_features = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
categorical_features = ['Type']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),  # To scale numerical features
        ('cat', OrdinalEncoder(categories=[['L', 'M', 'H']]), categorical_features)  # To encode categorical features
    ])

# Fit preprocessor on train
X_train_processed = preprocessor.fit_transform(X_train)  # Fit and transform
X_test_processed = preprocessor.transform(X_test)  # Transform

# Convert back to df for easier handling during feature selection
feature_names = numerical_features + categorical_features
X_train_processed = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_processed = pd.DataFrame(X_test_processed, columns=feature_names)

display(X_train_processed.head())

## 4. Feature Selection
Evaluate feature importance using mutual information. **Here we select the TOP 2 features** to restrict our downstream models.

In [ ]:
selector = SelectKBest(score_func=mutual_info_classif, k=2)
selector.fit(X_train_processed, y_train)

feature_scores = pd.DataFrame({'Feature': feature_names, 'Score': selector.scores_})
feature_scores = feature_scores.sort_values(by='Score', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x='Score', y='Feature', data=feature_scores)
plt.title('Feature Importance (Mutual Information)')
plt.show()

# Subsetting X_train and X_test to strictly the top 2 features
top_2_features = selector.get_feature_names_out()
print(f"\nSelected top 2 features: {list(top_2_features)}")

X_train_processed = pd.DataFrame(selector.transform(X_train_processed), columns=top_2_features)
X_test_processed = pd.DataFrame(selector.transform(X_test_processed), columns=top_2_features)

### 4.1 Feature Selection: Correlation Threshold (Alternative)
Use a Pearson correlation matrix to drop features that have an absolute correlation with the target variable below a specific threshold.

In [ ]:
# Temporarily join target to train features to compute correlation matrix
# Note: Since we subsetted X_train_processed to the top 2 features above, this will only show correlations for those 2 features unless we use the full DataFrame again.
train_full = X_train_processed.copy()
train_full[target_col] = y_train.values

corr_matrix = train_full.corr()
display(corr_matrix[[target_col]].sort_values(by=target_col, ascending=False))

threshold = 0.05
features_to_drop = []

for feature in X_train_processed.columns:
    if abs(corr_matrix[target_col][feature]) < threshold:
        features_to_drop.append(feature)

print(f"\nFeatures dropped due to abs(correlation) < {threshold}: {features_to_drop}")

# Apply dropping to train and test sets (uncomment to use this method for downstream models)
# X_train_processed = X_train_processed.drop(columns=features_to_drop)
# X_test_processed = X_test_processed.drop(columns=features_to_drop)

# 5. Model Tryouts & Cross-Validation
Since our data is sequential, we must use `TimeSeriesSplit` for cross-validation to prevent "future" records from being used to predict "past" records during hyperparameter tuning.

In [ ]:
# Define TimeSeriesSplit to prevent lookahead bias
tscv = TimeSeriesSplit(n_splits=3)

# Define parameter grids for tuning
param_grids = {
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=1000, random_state=42),
        'params': {'C': [0.1, 1.0, 10.0]}
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {'n_estimators': [50, 100], 'max_depth': [None, 5, 10]}
    },
    'Gradient Boosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {'n_estimators': [50, 100], 'learning_rate': [0.01, 0.1]}
    }
}

for name, config in param_grids.items():
    print(f"\n{'='*40}\n{name}\n{'='*40}")
    
    grid_search = GridSearchCV(config['model'], config['params'], cv=tscv, scoring='roc_auc', n_jobs=-1)
    grid_search.fit(X_train_processed, y_train)
    
    print(f"Best Parameters: {grid_search.best_params_}")
    
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test_processed)
    y_proba = best_model.predict_proba(X_test_processed)[:, 1]
    
    print("\nOut of Sample Test Results:")
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}\n")